In [0]:
# Project #4 — Supply Chain Planning & Operations Intelligence
# Silver Layer — Step 01: Verify Bronze Source Tables
#
# Purpose:
# Confirm that the validated Bronze Delta tables are available before
# beginning any Silver data-quality or business transformations.

bronze_catalog = "workspace"
bronze_schema = "bronze_sop"

bronze_tables = spark.sql(
    f"SHOW TABLES IN {bronze_catalog}.{bronze_schema}"
).select("tableName").collect()

bronze_table_names = sorted([row["tableName"] for row in bronze_tables])

print("SILVER TRANSFORMATION — BRONZE SOURCE VERIFICATION")
print("=" * 80)
print(f"Source schema: {bronze_catalog}.{bronze_schema}")
print(f"Bronze tables found: {len(bronze_table_names)}")
print()

for i, table_name in enumerate(bronze_table_names, start=1):
    print(f"{i:>2}. {table_name}")

print("=" * 80)

if len(bronze_table_names) == 21:
    print("SOURCE VERIFICATION PASSED: 21 Bronze Delta tables are available.")
else:
    raise ValueError(
        f"Expected 21 Bronze tables but found {len(bronze_table_names)}."
    )

SILVER TRANSFORMATION — BRONZE SOURCE VERIFICATION
Source schema: workspace.bronze_sop
Bronze tables found: 21

 1. alternative_machine
 2. attainment_parameter
 3. customer_order
 4. demand_plan
 5. finished_product_economics
 6. finished_product_type
 7. machine
 8. machine_capacity
 9. packing_capacity
10. packing_machine
11. planning_parameters
12. planning_periods
13. product_inventory
14. product_lot_size
15. product_packing_machine
16. product_production_alternative
17. product_raw_material_requirement
18. product_successor
19. raw_material
20. semi_finished_acquisition
21. semi_finished_product_cost
SOURCE VERIFICATION PASSED: 21 Bronze Delta tables are available.


In [0]:
# Project #4 — Supply Chain Planning & Operations Intelligence
# Silver Layer — Step 02: Profile Bronze Tables
#
# Purpose:
# Establish a baseline profile of every Bronze Delta table before
# applying Silver transformations.
#
# This step records:
# - Row count
# - Column count
# - Duplicate row count
# - Null cell count
#
# No source data is modified.

from pyspark.sql import functions as F

print("SILVER TRANSFORMATION — BRONZE BASELINE PROFILE")
print("=" * 115)

bronze_profile = []

for table_name in bronze_table_names:
    full_table_name = f"{bronze_catalog}.{bronze_schema}.{table_name}"
    df = spark.table(full_table_name)

    row_count = df.count()
    column_count = len(df.columns)

    distinct_row_count = df.distinct().count()
    duplicate_row_count = row_count - distinct_row_count

    null_expressions = [
        F.sum(F.when(F.col(f"`{column}`").isNull(), 1).otherwise(0))
        .alias(column)
        for column in df.columns
    ]

    null_counts = df.agg(*null_expressions).first().asDict()
    null_cell_count = sum(
        value for value in null_counts.values()
        if value is not None
    )

    bronze_profile.append({
        "table_name": table_name,
        "row_count": row_count,
        "column_count": column_count,
        "duplicate_rows": duplicate_row_count,
        "null_cells": null_cell_count
    })

    print(
        f"{table_name:<40} | "
        f"Rows: {row_count:<5} | "
        f"Cols: {column_count:<3} | "
        f"Duplicate Rows: {duplicate_row_count:<5} | "
        f"Null Cells: {null_cell_count}"
    )

print("=" * 115)
print(f"Tables profiled: {len(bronze_profile)}")

total_rows = sum(item["row_count"] for item in bronze_profile)

print(f"Total Bronze records profiled: {total_rows}")

SILVER TRANSFORMATION — BRONZE BASELINE PROFILE
alternative_machine                      | Rows: 734   | Cols: 3   | Duplicate Rows: 0     | Null Cells: 0
attainment_parameter                     | Rows: 3     | Cols: 3   | Duplicate Rows: 0     | Null Cells: 0
customer_order                           | Rows: 61    | Cols: 5   | Duplicate Rows: 0     | Null Cells: 0
demand_plan                              | Rows: 246   | Cols: 4   | Duplicate Rows: 0     | Null Cells: 0
finished_product_economics               | Rows: 67    | Cols: 4   | Duplicate Rows: 0     | Null Cells: 0
finished_product_type                    | Rows: 67    | Cols: 2   | Duplicate Rows: 0     | Null Cells: 0
machine                                  | Rows: 39    | Cols: 2   | Duplicate Rows: 0     | Null Cells: 0
machine_capacity                         | Rows: 312   | Cols: 4   | Duplicate Rows: 0     | Null Cells: 0
packing_capacity                         | Rows: 56    | Cols: 4   | Duplicate Rows: 0     | Nul

In [0]:
# Project #4 — Supply Chain Planning & Operations Intelligence
# Silver Layer — Step 03: Profile Blank and Whitespace String Values
#
# Purpose:
# Detect missing-like values represented as empty strings or whitespace
# in Bronze string columns.
#
# No source data is modified.

from pyspark.sql.types import StringType
from pyspark.sql import functions as F

print("SILVER TRANSFORMATION — BLANK / WHITESPACE PROFILE")
print("=" * 105)

blank_profile = []
total_blank_values = 0
string_columns_checked = 0

for table_name in bronze_table_names:
    full_table_name = f"{bronze_catalog}.{bronze_schema}.{table_name}"
    df = spark.table(full_table_name)

    string_columns = [
        field.name
        for field in df.schema.fields
        if isinstance(field.dataType, StringType)
    ]

    for column_name in string_columns:
        string_columns_checked += 1

        blank_count = (
            df.filter(
                F.col(f"`{column_name}`").isNotNull()
                & (F.trim(F.col(f"`{column_name}`")) == "")
            )
            .count()
        )

        blank_profile.append({
            "table_name": table_name,
            "column_name": column_name,
            "blank_or_whitespace_count": blank_count
        })

        total_blank_values += blank_count

        print(
            f"{table_name:<40} | "
            f"{column_name:<55} | "
            f"Blank/Whitespace: {blank_count}"
        )

print("=" * 105)
print(f"String columns checked: {string_columns_checked}")
print(f"Total blank/whitespace values found: {total_blank_values}")

SILVER TRANSFORMATION — BLANK / WHITESPACE PROFILE
customer_order                           | finished_product                                        | Blank/Whitespace: 0
demand_plan                              | finished_product                                        | Blank/Whitespace: 0
finished_product_economics               | finished_product                                        | Blank/Whitespace: 0
finished_product_type                    | finished_product                                        | Blank/Whitespace: 0
finished_product_type                    | mts_mto                                                 | Blank/Whitespace: 0
planning_parameters                      | parameter_name                                          | Blank/Whitespace: 0
product_inventory                        | product                                                 | Blank/Whitespace: 0
product_lot_size                         | product                                                 | B

In [0]:
# Project #4 — Supply Chain Planning & Operations Intelligence
# Silver Layer — Step 04: Candidate Business-Key Uniqueness Profile
#
# Purpose:
# Test candidate business keys for uniqueness within Bronze tables.
# Candidate keys are based on the apparent grain of each source block.
#
# This is a diagnostic profile only.
# No primary-key constraints are declared and no data is modified.

from pyspark.sql import functions as F

candidate_keys = {
    "planning_parameters": ["parameter_name"],
    "planning_periods": ["period"],
    "finished_product_type": ["finished_product"],
    "finished_product_economics": ["finished_product"],
    "semi_finished_product_cost": ["semi_finished_product"],
    "product_inventory": ["product"],
    "product_raw_material_requirement": ["product"],
    "customer_order": ["order"],
    "demand_plan": ["finished_product", "period"],
    "product_lot_size": ["product"],
    "product_production_alternative": ["product", "alternative"],
    "alternative_machine": ["alternative", "machine"],
    "machine": ["machine"],
    "machine_capacity": ["machine", "period"],
    "packing_machine": ["packing_machine"],
    "product_packing_machine": ["finished_products", "packing_machine"],
    "packing_capacity": ["packing_machine", "period"],
    "semi_finished_acquisition": [
        "semi_finished_products_that_can_be_acquired_from_the_market"
    ],
    "raw_material": ["raw_material"]
}

print("SILVER TRANSFORMATION — CANDIDATE BUSINESS-KEY UNIQUENESS PROFILE")
print("=" * 120)

candidate_key_profile = []

for table_name, key_columns in candidate_keys.items():

    full_table_name = f"{bronze_catalog}.{bronze_schema}.{table_name}"
    df = spark.table(full_table_name)

    row_count = df.count()

    distinct_key_count = (
        df.select(*[F.col(f"`{c}`") for c in key_columns])
        .distinct()
        .count()
    )

    duplicate_key_rows = row_count - distinct_key_count

    status = "PASS" if duplicate_key_rows == 0 else "REVIEW"

    candidate_key_profile.append({
        "table_name": table_name,
        "candidate_key": key_columns,
        "row_count": row_count,
        "distinct_key_count": distinct_key_count,
        "duplicate_key_rows": duplicate_key_rows,
        "status": status
    })

    print(
        f"{table_name:<40} | "
        f"Key: {', '.join(key_columns):<40} | "
        f"Rows: {row_count:<5} | "
        f"Distinct Keys: {distinct_key_count:<5} | "
        f"Duplicate Key Rows: {duplicate_key_rows:<5} | "
        f"{status}"
    )

print("=" * 120)

passed = sum(
    1 for item in candidate_key_profile
    if item["status"] == "PASS"
)

review = len(candidate_key_profile) - passed

print(f"Candidate keys tested: {len(candidate_key_profile)}")
print(f"PASS: {passed}")
print(f"REVIEW: {review}")

SILVER TRANSFORMATION — CANDIDATE BUSINESS-KEY UNIQUENESS PROFILE
planning_parameters                      | Key: parameter_name                           | Rows: 7     | Distinct Keys: 7     | Duplicate Key Rows: 0     | PASS
planning_periods                         | Key: period                                   | Rows: 8     | Distinct Keys: 8     | Duplicate Key Rows: 0     | PASS
finished_product_type                    | Key: finished_product                         | Rows: 67    | Distinct Keys: 67    | Duplicate Key Rows: 0     | PASS
finished_product_economics               | Key: finished_product                         | Rows: 67    | Distinct Keys: 67    | Duplicate Key Rows: 0     | PASS
semi_finished_product_cost               | Key: semi_finished_product                    | Rows: 225   | Distinct Keys: 225   | Duplicate Key Rows: 0     | PASS
product_inventory                        | Key: product                                  | Rows: 292   | Distinct Keys: 292   | D

In [0]:
# Project #4 — Supply Chain Planning & Operations Intelligence
# Silver Layer — Step 05: Investigate Remaining Table Grain
#
# Purpose:
# Examine the two Bronze tables not included in the initial candidate-key
# profile and determine their observed record grain from the source data.
#
# No keys are declared and no source data is modified.

from pyspark.sql import functions as F

print("SILVER TRANSFORMATION — REMAINING TABLE GRAIN INVESTIGATION")
print("=" * 100)

# -------------------------------------------------------------------
# 1. product_successor
# -------------------------------------------------------------------

product_successor_df = spark.table(
    f"{bronze_catalog}.{bronze_schema}.product_successor"
)

print("\nTABLE: product_successor")
print("-" * 100)

print(f"Rows: {product_successor_df.count()}")
print(
    "Distinct products:",
    product_successor_df.select("product").distinct().count()
)
print(
    "Distinct successor products:",
    product_successor_df.select("successor_product").distinct().count()
)
print(
    "Distinct product + successor combinations:",
    product_successor_df
        .select("product", "successor_product")
        .distinct()
        .count()
)

product_successor_duplicate_pairs = (
    product_successor_df
    .groupBy("product", "successor_product")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Repeated product + successor combinations:",
    product_successor_duplicate_pairs.count()
)

print("\nSample product_successor records:")
product_successor_df.show(10, truncate=False)


# -------------------------------------------------------------------
# 2. attainment_parameter
# -------------------------------------------------------------------

attainment_df = spark.table(
    f"{bronze_catalog}.{bronze_schema}.attainment_parameter"
)

print("\nTABLE: attainment_parameter")
print("-" * 100)

print(f"Rows: {attainment_df.count()}")
print(
    "Distinct levels_of_attainment:",
    attainment_df.select("levels_of_attainment").distinct().count()
)
print(
    "Distinct percentage_of_attainment:",
    attainment_df.select("percentage_of_attainment").distinct().count()
)

print("\nAll attainment_parameter records:")
attainment_df.orderBy("levels_of_attainment").show(truncate=False)

print("=" * 100)
print("GRAIN INVESTIGATION COMPLETE")

SILVER TRANSFORMATION — REMAINING TABLE GRAIN INVESTIGATION

TABLE: product_successor
----------------------------------------------------------------------------------------------------
Rows: 1026
Distinct products: 292
Distinct successor products: 292
Distinct product + successor combinations: 1026
Repeated product + successor combinations: 0

Sample product_successor records:
+--------+-----------------+-----------------------------------------+
|product |successor_product|quantity_of_product_used_in_successor_m_m|
+--------+-----------------+-----------------------------------------+
|8082258B|8384095          |18                                       |
|8082258B|8384184          |34                                       |
|8082258B|8384186          |34                                       |
|8082258B|8384187          |34                                       |
|8082276C|7207088          |44                                       |
|8082276C|8051020Z         |3                     

In [0]:
# Project #4 — Supply Chain Planning & Operations Intelligence
# Silver Layer — Step 06: Referential Integrity Profile
#
# Purpose:
# Test observed relationships between Bronze tables and identify
# foreign-key-like values that do not have a matching parent/reference value.
#
# This is diagnostic only.
# No constraints are declared and no source data is modified.

from pyspark.sql import functions as F

bronze_catalog = "workspace"
bronze_schema = "bronze_sop"

referential_tests = [
    # Child table, child column, parent table, parent column

    ("customer_order", "finished_product",
     "finished_product_type", "finished_product"),

    ("demand_plan", "finished_product",
     "finished_product_type", "finished_product"),

    ("finished_product_economics", "finished_product",
     "finished_product_type", "finished_product"),

    ("machine_capacity", "machine",
     "machine", "machine"),

    ("alternative_machine", "machine",
     "machine", "machine"),

    ("packing_capacity", "packing_machine",
     "packing_machine", "packing_machine"),

    ("product_packing_machine", "packing_machine",
     "packing_machine", "packing_machine"),

    ("demand_plan", "period",
     "planning_periods", "period"),

    ("machine_capacity", "period",
     "planning_periods", "period"),

    ("packing_capacity", "period",
     "planning_periods", "period"),

    ("product_production_alternative", "product",
     "product_inventory", "product"),

    ("product_raw_material_requirement", "product",
     "product_inventory", "product"),

    ("product_lot_size", "product",
     "product_inventory", "product"),

    ("product_successor", "product",
     "product_inventory", "product"),

    ("product_successor", "successor_product",
     "product_inventory", "product"),

    ("product_packing_machine", "finished_products",
     "finished_product_type", "finished_product"),
]

print("SILVER TRANSFORMATION — REFERENTIAL INTEGRITY PROFILE")
print("=" * 125)

referential_profile = []

for (
    child_table,
    child_column,
    parent_table,
    parent_column
) in referential_tests:

    child_df = spark.table(
        f"{bronze_catalog}.{bronze_schema}.{child_table}"
    )

    parent_df = spark.table(
        f"{bronze_catalog}.{bronze_schema}.{parent_table}"
    )

    child_values = (
        child_df
        .select(F.col(f"`{child_column}`").alias("key_value"))
        .distinct()
    )

    parent_values = (
        parent_df
        .select(F.col(f"`{parent_column}`").alias("key_value"))
        .distinct()
    )

    orphan_values = child_values.join(
        parent_values,
        on="key_value",
        how="left_anti"
    )

    distinct_child_values = child_values.count()
    orphan_value_count = orphan_values.count()

    status = "PASS" if orphan_value_count == 0 else "REVIEW"

    referential_profile.append({
        "child_table": child_table,
        "child_column": child_column,
        "parent_table": parent_table,
        "parent_column": parent_column,
        "distinct_child_values": distinct_child_values,
        "orphan_values": orphan_value_count,
        "status": status
    })

    relationship = (
        f"{child_table}.{child_column} -> "
        f"{parent_table}.{parent_column}"
    )

    print(
        f"{relationship:<90} | "
        f"Child Values: {distinct_child_values:<5} | "
        f"Orphans: {orphan_value_count:<5} | "
        f"{status}"
    )

print("=" * 125)

passed = sum(
    1 for item in referential_profile
    if item["status"] == "PASS"
)

review = len(referential_profile) - passed

print(f"Relationships tested: {len(referential_profile)}")
print(f"PASS: {passed}")
print(f"REVIEW: {review}")

SILVER TRANSFORMATION — REFERENTIAL INTEGRITY PROFILE
customer_order.finished_product -> finished_product_type.finished_product                  | Child Values: 43    | Orphans: 0     | PASS
demand_plan.finished_product -> finished_product_type.finished_product                     | Child Values: 41    | Orphans: 0     | PASS
finished_product_economics.finished_product -> finished_product_type.finished_product      | Child Values: 67    | Orphans: 0     | PASS
machine_capacity.machine -> machine.machine                                                | Child Values: 39    | Orphans: 0     | PASS
alternative_machine.machine -> machine.machine                                             | Child Values: 39    | Orphans: 0     | PASS
packing_capacity.packing_machine -> packing_machine.packing_machine                        | Child Values: 7     | Orphans: 0     | PASS
product_packing_machine.packing_machine -> packing_machine.packing_machine                 | Child Values: 7     | Orphans: 

In [0]:
# Project #4 — Supply Chain Planning & Operations Intelligence
# Silver Layer — Step 07: Domain Values and Numeric Range Profile
#
# Purpose:
# Inspect important categorical domains and numeric ranges before
# defining Silver business transformations or validation rules.
#
# No values are changed and no business meanings are inferred.

from pyspark.sql import functions as F

bronze_catalog = "workspace"
bronze_schema = "bronze_sop"

print("SILVER TRANSFORMATION — DOMAIN VALUES AND NUMERIC RANGE PROFILE")
print("=" * 110)

# -------------------------------------------------------------------
# 1. Categorical / coded domains
# -------------------------------------------------------------------

categorical_checks = [
    ("finished_product_type", "mts_mto"),
    ("product_production_alternative", "main_alternative"),
]

print("\n1. CATEGORICAL / CODED DOMAINS")
print("-" * 110)

for table_name, column_name in categorical_checks:

    df = spark.table(
        f"{bronze_catalog}.{bronze_schema}.{table_name}"
    )

    print(f"\n{table_name}.{column_name}")

    (
        df.groupBy(F.col(f"`{column_name}`"))
          .count()
          .orderBy(F.col(f"`{column_name}`"))
          .show(truncate=False)
    )


# -------------------------------------------------------------------
# 2. Numeric ranges
# -------------------------------------------------------------------

numeric_checks = [
    ("customer_order",
     "due_date_days_in_the_planning_horizon"),

    ("customer_order",
     "quantity_m"),

    ("customer_order",
     "sales_value_eur"),

    ("finished_product_economics",
     "industrial_cost_eur_m"),

    ("finished_product_economics",
     "margin_pct"),

    ("finished_product_economics",
     "average_sales_price_eur_m"),

    ("attainment_parameter",
     "percentage_of_attainment"),

    ("attainment_parameter",
     "premium_coefficient"),

    ("planning_periods",
     "period"),

    ("planning_periods",
     "number_of_days"),
]

print("\n2. NUMERIC RANGE PROFILE")
print("-" * 110)

for table_name, column_name in numeric_checks:

    df = spark.table(
        f"{bronze_catalog}.{bronze_schema}.{table_name}"
    )

    result = (
        df.agg(
            F.count(F.col(f"`{column_name}`")).alias("count"),
            F.min(F.col(f"`{column_name}`")).alias("min"),
            F.max(F.col(f"`{column_name}`")).alias("max"),
            F.avg(F.col(f"`{column_name}`")).alias("avg")
        )
        .first()
    )

    print(
        f"{table_name:<35} | "
        f"{column_name:<45} | "
        f"Count: {result['count']:<5} | "
        f"Min: {result['min']} | "
        f"Max: {result['max']} | "
        f"Avg: {result['avg']}"
    )


# -------------------------------------------------------------------
# 3. Due-date sign distribution
# -------------------------------------------------------------------

customer_order_df = spark.table(
    f"{bronze_catalog}.{bronze_schema}.customer_order"
)

due_date_column = "due_date_days_in_the_planning_horizon"

due_date_sign_profile = (
    customer_order_df
    .select(
        F.when(F.col(due_date_column) < 0, "Negative")
         .when(F.col(due_date_column) == 0, "Zero")
         .otherwise("Positive")
         .alias("due_date_sign")
    )
    .groupBy("due_date_sign")
    .count()
    .orderBy("due_date_sign")
)

print("\n3. CUSTOMER ORDER DUE-DATE SIGN DISTRIBUTION")
print("-" * 110)

due_date_sign_profile.show(truncate=False)

print("=" * 110)
print("DOMAIN AND NUMERIC RANGE PROFILING COMPLETE")

SILVER TRANSFORMATION — DOMAIN VALUES AND NUMERIC RANGE PROFILE

1. CATEGORICAL / CODED DOMAINS
--------------------------------------------------------------------------------------------------------------

finished_product_type.mts_mto
+-------+-----+
|mts_mto|count|
+-------+-----+
|MTO    |26   |
|MTS    |41   |
+-------+-----+


product_production_alternative.main_alternative
+----------------+-----+
|main_alternative|count|
+----------------+-----+
|1               |291  |
|2               |301  |
+----------------+-----+


2. NUMERIC RANGE PROFILE
--------------------------------------------------------------------------------------------------------------
customer_order                      | due_date_days_in_the_planning_horizon         | Count: 61    | Min: -7 | Max: 101 | Avg: 29.65573770491803
customer_order                      | quantity_m                                    | Count: 61    | Min: 602 | Max: 80000 | Avg: 14986.88524590164
customer_order                     

In [0]:
# Project #4 — Supply Chain Planning & Operations Intelligence
# Silver Layer — Step 08: Due-Date and Planning-Horizon Validation
#
# Purpose:
# Compare customer-order due dates with source-derived planning-horizon
# parameters and quantify records before, within, or beyond the horizon.
#
# No due dates are corrected, removed, capped, or otherwise modified.

from pyspark.sql import functions as F

bronze_catalog = "workspace"
bronze_schema = "bronze_sop"

parameters_df = spark.table(
    f"{bronze_catalog}.{bronze_schema}.planning_parameters"
)

orders_df = spark.table(
    f"{bronze_catalog}.{bronze_schema}.customer_order"
)

# Retrieve source-derived planning parameters.
planning_parameter_values = {
    row["parameter_name"]: row["parameter_value"]
    for row in parameters_df.collect()
}

number_of_periods = planning_parameter_values.get("Number of periods")
total_horizon_days = planning_parameter_values.get("Total number of days")

print("SILVER TRANSFORMATION — DUE-DATE / PLANNING-HORIZON VALIDATION")
print("=" * 105)

print(f"Source parameter — Number of periods: {number_of_periods}")
print(f"Source parameter — Total number of days: {total_horizon_days}")

if total_horizon_days is None:
    raise ValueError(
        "Source parameter 'Total number of days' was not found."
    )

# Classify orders relative to the source-defined planning horizon.
order_horizon_profile = (
    orders_df
    .withColumn(
        "horizon_status",
        F.when(
            F.col("due_date_days_in_the_planning_horizon") < 0,
            "Before horizon start"
        )
        .when(
            F.col("due_date_days_in_the_planning_horizon")
            <= F.lit(total_horizon_days),
            "Within horizon"
        )
        .otherwise("Beyond horizon end")
    )
)

print("\nORDER DISTRIBUTION RELATIVE TO PLANNING HORIZON")
print("-" * 105)

(
    order_horizon_profile
    .groupBy("horizon_status")
    .count()
    .orderBy("horizon_status")
    .show(truncate=False)
)

print("\nORDERS BEFORE HORIZON START")
print("-" * 105)

(
    order_horizon_profile
    .filter(F.col("horizon_status") == "Before horizon start")
    .select(
        "order",
        "finished_product",
        "due_date_days_in_the_planning_horizon",
        "quantity_m",
        "sales_value_eur"
    )
    .orderBy("due_date_days_in_the_planning_horizon", "order")
    .show(truncate=False)
)

beyond_horizon_count = (
    order_horizon_profile
    .filter(F.col("horizon_status") == "Beyond horizon end")
    .count()
)

print("=" * 105)
print(f"Orders beyond planning-horizon end: {beyond_horizon_count}")
print("DUE-DATE / PLANNING-HORIZON VALIDATION COMPLETE")

SILVER TRANSFORMATION — DUE-DATE / PLANNING-HORIZON VALIDATION
Source parameter — Number of periods: 8.0
Source parameter — Total number of days: 101.0

ORDER DISTRIBUTION RELATIVE TO PLANNING HORIZON
---------------------------------------------------------------------------------------------------------
+--------------------+-----+
|horizon_status      |count|
+--------------------+-----+
|Before horizon start|9    |
|Within horizon      |52   |
+--------------------+-----+


ORDERS BEFORE HORIZON START
---------------------------------------------------------------------------------------------------------
+-----+----------------+-------------------------------------+----------+---------------+
|order|finished_product|due_date_days_in_the_planning_horizon|quantity_m|sales_value_eur|
+-----+----------------+-------------------------------------+----------+---------------+
|2    |6182513         |-7                                   |6000      |12120.0        |
|4    |6182515         

In [0]:
# Project #4 — Supply Chain Planning & Operations Intelligence
# Silver Layer — Step 09: Validate Planning-Period Structure
#
# Purpose:
# Validate the planning-period calendar against source-derived planning
# parameters and inspect period sequencing and horizon boundaries.
#
# No source data is modified.

from pyspark.sql import functions as F

bronze_catalog = "workspace"
bronze_schema = "bronze_sop"

periods_df = spark.table(
    f"{bronze_catalog}.{bronze_schema}.planning_periods"
)

parameters_df = spark.table(
    f"{bronze_catalog}.{bronze_schema}.planning_parameters"
)

parameter_values = {
    row["parameter_name"]: row["parameter_value"]
    for row in parameters_df.collect()
}

expected_periods = int(parameter_values["Number of periods"])
expected_horizon_days = int(parameter_values["Total number of days"])

actual_period_count = periods_df.count()

period_stats = (
    periods_df
    .agg(
        F.min("period").alias("min_period"),
        F.max("period").alias("max_period"),
        F.sum("number_of_days").alias("sum_period_days"),
        F.max("last_day_of_period").alias("max_last_day")
    )
    .first()
)

distinct_periods = periods_df.select("period").distinct().count()

print("SILVER TRANSFORMATION — PLANNING-PERIOD STRUCTURE VALIDATION")
print("=" * 105)

print(f"Expected number of periods:       {expected_periods}")
print(f"Actual period rows:               {actual_period_count}")
print(f"Distinct period values:           {distinct_periods}")
print(f"Minimum period:                   {period_stats['min_period']}")
print(f"Maximum period:                   {period_stats['max_period']}")
print()

print(f"Expected planning-horizon days:   {expected_horizon_days}")
print(f"Sum of period number_of_days:     {period_stats['sum_period_days']}")
print(f"Maximum last_day_of_period:       {period_stats['max_last_day']}")

print("\nPLANNING PERIOD DETAIL")
print("-" * 105)

(
    periods_df
    .orderBy("period")
    .show(truncate=False)
)

period_count_pass = (
    actual_period_count == expected_periods
    and distinct_periods == expected_periods
)

period_sequence_pass = (
    period_stats["min_period"] == 1
    and period_stats["max_period"] == expected_periods
)

day_total_pass = (
    period_stats["sum_period_days"] == expected_horizon_days
)

horizon_end_pass = (
    period_stats["max_last_day"] == expected_horizon_days
)

print("=" * 105)
print(
    f"Period count reconciliation:      "
    f"{'PASS' if period_count_pass else 'REVIEW'}"
)
print(
    f"Period sequence validation:       "
    f"{'PASS' if period_sequence_pass else 'REVIEW'}"
)
print(
    f"Period-day total reconciliation:  "
    f"{'PASS' if day_total_pass else 'REVIEW'}"
)
print(
    f"Horizon-end reconciliation:       "
    f"{'PASS' if horizon_end_pass else 'REVIEW'}"
)

if all([
    period_count_pass,
    period_sequence_pass,
    day_total_pass,
    horizon_end_pass
]):
    print(
        "PLANNING-PERIOD STRUCTURE VALIDATION PASSED."
    )
else:
    print(
        "PLANNING-PERIOD STRUCTURE REQUIRES REVIEW."
    )

SILVER TRANSFORMATION — PLANNING-PERIOD STRUCTURE VALIDATION
Expected number of periods:       8
Actual period rows:               8
Distinct period values:           8
Minimum period:                   1
Maximum period:                   8

Expected planning-horizon days:   101
Sum of period number_of_days:     101
Maximum last_day_of_period:       101

PLANNING PERIOD DETAIL
---------------------------------------------------------------------------------------------------------
+------+-----------------+------------------+--------------+
|period|macroperiod_month|last_day_of_period|number_of_days|
+------+-----------------+------------------+--------------+
|1     |1                |7                 |7             |
|2     |1                |15                |8             |
|3     |2                |21                |6             |
|4     |2                |28                |7             |
|5     |2                |35                |7             |
|6     |2                |

In [0]:
# Project #4 — Supply Chain Planning & Operations Intelligence
# Silver Layer — Step 10: Define Silver Transformation Contract
#
# Purpose:
# Define controlled Bronze-to-Silver naming transformations and
# source-supported derived fields before Silver tables are created.
#
# Bronze tables remain unchanged.

bronze_catalog = "workspace"
bronze_schema = "bronze_sop"

silver_catalog = "workspace"
silver_schema = "silver_sop"

# ---------------------------------------------------------------
# Controlled column renaming
# ---------------------------------------------------------------

silver_column_renames = {

    "customer_order": {
        "order": "order_id"
    },

    "product_packing_machine": {
        "finished_products": "finished_product"
    },

    "demand_plan": {
        "total_demand_m_adjusted_if_orders_are_greater_than_forecasts":
            "adjusted_total_demand_m"
    },

    "semi_finished_acquisition": {
        "semi_finished_products_that_can_be_acquired_from_the_market":
            "semi_finished_product",

        "maximum_purchasing_quantity_per_macroperiod_month_m":
            "max_purchase_quantity_per_month_m"
    },

    "raw_material": {
        "maximum_purchasing_quantity_per_macroperiod_month_kg":
            "max_purchase_quantity_per_month_kg"
    }
}

# ---------------------------------------------------------------
# Source-supported derived fields
# ---------------------------------------------------------------

silver_derived_fields = {

    "customer_order": [
        "planning_horizon_status"
    ]
}

# ---------------------------------------------------------------
# Explicit preservation rules
# ---------------------------------------------------------------

preservation_rules = {
    "customer_order.due_date_days_in_the_planning_horizon":
        "Preserve source values, including -7 pre-horizon records.",

    "product_production_alternative.main_alternative":
        "Preserve source codes 1 and 2; business labels are not evidenced.",

    "finished_product_type.mts_mto":
        "Preserve observed source values MTS and MTO."
}

print("SILVER TRANSFORMATION CONTRACT")
print("=" * 100)

print(f"Bronze source: {bronze_catalog}.{bronze_schema}")
print(f"Silver target: {silver_catalog}.{silver_schema}")

print("\nCONTROLLED COLUMN RENAMES")
print("-" * 100)

rename_count = 0

for table_name, mappings in silver_column_renames.items():
    for source_column, silver_column in mappings.items():

        rename_count += 1

        print(
            f"{table_name:<35} | "
            f"{source_column} -> {silver_column}"
        )

print("\nSOURCE-SUPPORTED DERIVED FIELDS")
print("-" * 100)

for table_name, fields in silver_derived_fields.items():
    for field in fields:
        print(f"{table_name:<35} | {field}")

print("\nEXPLICIT PRESERVATION RULES")
print("-" * 100)

for field_name, rule in preservation_rules.items():
    print(f"{field_name}")
    print(f"  -> {rule}")

print("=" * 100)
print(f"Controlled renames defined: {rename_count}")
print(
    "SILVER CONTRACT STATUS: DEFINED — "
    "no Bronze data has been modified."
)

SILVER TRANSFORMATION CONTRACT
Bronze source: workspace.bronze_sop
Silver target: workspace.silver_sop

CONTROLLED COLUMN RENAMES
----------------------------------------------------------------------------------------------------
customer_order                      | order -> order_id
product_packing_machine             | finished_products -> finished_product
demand_plan                         | total_demand_m_adjusted_if_orders_are_greater_than_forecasts -> adjusted_total_demand_m
semi_finished_acquisition           | semi_finished_products_that_can_be_acquired_from_the_market -> semi_finished_product
semi_finished_acquisition           | maximum_purchasing_quantity_per_macroperiod_month_m -> max_purchase_quantity_per_month_m
raw_material                        | maximum_purchasing_quantity_per_macroperiod_month_kg -> max_purchase_quantity_per_month_kg

SOURCE-SUPPORTED DERIVED FIELDS
---------------------------------------------------------------------------------------------------

In [0]:
# Project #4 — Supply Chain Planning & Operations Intelligence
# Silver Layer — Step 11: Create Silver Schema and Build Transformed DataFrames
#
# Purpose:
# Create the Silver target schema and construct standardized Silver
# DataFrames directly from the persisted Bronze Delta tables.
#
# This step is self-contained and does NOT write Silver Delta tables yet.

from pyspark.sql import functions as F

bronze_catalog = "workspace"
bronze_schema = "bronze_sop"

silver_catalog = "workspace"
silver_schema = "silver_sop"


# ---------------------------------------------------------------
# 1. Rebuild Bronze table inventory from Unity Catalog
# ---------------------------------------------------------------

bronze_tables = spark.sql(
    f"SHOW TABLES IN {bronze_catalog}.{bronze_schema}"
).select("tableName").collect()

bronze_table_names = sorted(
    [row["tableName"] for row in bronze_tables]
)

if len(bronze_table_names) != 21:
    raise ValueError(
        f"Expected 21 Bronze tables but found "
        f"{len(bronze_table_names)}."
    )


# ---------------------------------------------------------------
# 2. Rebuild controlled Silver transformation contract
# ---------------------------------------------------------------

silver_column_renames = {

    "customer_order": {
        "order": "order_id"
    },

    "product_packing_machine": {
        "finished_products": "finished_product"
    },

    "demand_plan": {
        "total_demand_m_adjusted_if_orders_are_greater_than_forecasts":
            "adjusted_total_demand_m"
    },

    "semi_finished_acquisition": {
        "semi_finished_products_that_can_be_acquired_from_the_market":
            "semi_finished_product",

        "maximum_purchasing_quantity_per_macroperiod_month_m":
            "max_purchase_quantity_per_month_m"
    },

    "raw_material": {
        "maximum_purchasing_quantity_per_macroperiod_month_kg":
            "max_purchase_quantity_per_month_kg"
    }
}


# ---------------------------------------------------------------
# 3. Create Silver schema
# ---------------------------------------------------------------

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"{silver_catalog}.{silver_schema}"
)


# ---------------------------------------------------------------
# 4. Retrieve source-defined planning horizon
# ---------------------------------------------------------------

parameters_df = spark.table(
    f"{bronze_catalog}.{bronze_schema}.planning_parameters"
)

planning_parameter_values = {
    row["parameter_name"]: row["parameter_value"]
    for row in parameters_df.collect()
}

total_horizon_days = planning_parameter_values.get(
    "Total number of days"
)

if total_horizon_days is None:
    raise ValueError(
        "Source parameter 'Total number of days' was not found."
    )


# ---------------------------------------------------------------
# 5. Build Silver DataFrames
# ---------------------------------------------------------------

silver_dataframes = {}

for table_name in bronze_table_names:

    df = spark.table(
        f"{bronze_catalog}.{bronze_schema}.{table_name}"
    )

    # Apply controlled renames.
    if table_name in silver_column_renames:

        for source_column, silver_column in (
            silver_column_renames[table_name].items()
        ):
            df = df.withColumnRenamed(
                source_column,
                silver_column
            )

    # Add source-supported analytical field.
    if table_name == "customer_order":

        df = df.withColumn(
            "planning_horizon_status",

            F.when(
                F.col(
                    "due_date_days_in_the_planning_horizon"
                ) < 0,
                F.lit("Before horizon start")
            )

            .when(
                F.col(
                    "due_date_days_in_the_planning_horizon"
                ) <= F.lit(total_horizon_days),
                F.lit("Within horizon")
            )

            .otherwise(
                F.lit("Beyond horizon end")
            )
        )

    silver_dataframes[table_name] = df


# ---------------------------------------------------------------
# 6. Validate and display transformation summary
# ---------------------------------------------------------------

print("SILVER TRANSFORMATION — BUILD STANDARDIZED DATAFRAMES")
print("=" * 110)

print(f"Bronze source: {bronze_catalog}.{bronze_schema}")
print(f"Silver target: {silver_catalog}.{silver_schema}")
print(f"Bronze tables discovered: {len(bronze_table_names)}")
print()

total_bronze_rows = 0
total_silver_rows = 0

for table_name in bronze_table_names:

    bronze_df = spark.table(
        f"{bronze_catalog}.{bronze_schema}.{table_name}"
    )

    silver_df = silver_dataframes[table_name]

    bronze_rows = bronze_df.count()
    silver_rows = silver_df.count()

    total_bronze_rows += bronze_rows
    total_silver_rows += silver_rows

    row_status = (
        "PASS"
        if bronze_rows == silver_rows
        else "REVIEW"
    )

    print(
        f"{table_name:<40} | "
        f"Bronze Cols: {len(bronze_df.columns):<3} | "
        f"Silver Cols: {len(silver_df.columns):<3} | "
        f"Bronze Rows: {bronze_rows:<5} | "
        f"Silver Rows: {silver_rows:<5} | "
        f"{row_status}"
    )

print("=" * 110)

print(
    f"Silver DataFrames constructed: {len(silver_dataframes)}"
)

print(
    f"Total Bronze records:          {total_bronze_rows}"
)

print(
    f"Total Silver records:          {total_silver_rows}"
)

if (
    len(silver_dataframes) == 21
    and total_bronze_rows == 4559
    and total_silver_rows == total_bronze_rows
):
    print(
        "TRANSFORMATION BUILD PASSED: "
        "21 Silver DataFrames constructed with "
        "full row-count reconciliation."
    )
else:
    print(
        "TRANSFORMATION BUILD REQUIRES REVIEW."
    )

print(
    "Silver Delta tables have NOT yet been persisted."
)

SILVER TRANSFORMATION — BUILD STANDARDIZED DATAFRAMES
Bronze source: workspace.bronze_sop
Silver target: workspace.silver_sop
Bronze tables discovered: 21

alternative_machine                      | Bronze Cols: 3   | Silver Cols: 3   | Bronze Rows: 734   | Silver Rows: 734   | PASS
attainment_parameter                     | Bronze Cols: 3   | Silver Cols: 3   | Bronze Rows: 3     | Silver Rows: 3     | PASS
customer_order                           | Bronze Cols: 5   | Silver Cols: 6   | Bronze Rows: 61    | Silver Rows: 61    | PASS
demand_plan                              | Bronze Cols: 4   | Silver Cols: 4   | Bronze Rows: 246   | Silver Rows: 246   | PASS
finished_product_economics               | Bronze Cols: 4   | Silver Cols: 4   | Bronze Rows: 67    | Silver Rows: 67    | PASS
finished_product_type                    | Bronze Cols: 2   | Silver Cols: 2   | Bronze Rows: 67    | Silver Rows: 67    | PASS
machine                                  | Bronze Cols: 2   | Silver Cols: 2

In [0]:
# Project #4 — Supply Chain Planning & Operations Intelligence
# Silver Layer — Step 12: Validate Silver Transformation Contract
#
# Purpose:
# Confirm that the in-memory Silver DataFrames implement the approved
# transformation contract before any Silver Delta tables are persisted.
#
# No tables are written in this step.

from pyspark.sql import functions as F

print("SILVER TRANSFORMATION — CONTRACT VALIDATION")
print("=" * 115)

validation_results = []


# ---------------------------------------------------------------
# 1. Validate controlled column renames
# ---------------------------------------------------------------

print("\n1. CONTROLLED COLUMN RENAME VALIDATION")
print("-" * 115)

for table_name, mappings in silver_column_renames.items():

    silver_columns = silver_dataframes[table_name].columns

    for bronze_column, silver_column in mappings.items():

        new_exists = silver_column in silver_columns
        old_removed = bronze_column not in silver_columns

        status = (
            "PASS"
            if new_exists and old_removed
            else "REVIEW"
        )

        validation_results.append(status)

        print(
            f"{table_name:<35} | "
            f"{bronze_column} -> {silver_column:<40} | "
            f"{status}"
        )


# ---------------------------------------------------------------
# 2. Validate customer-order derived field
# ---------------------------------------------------------------

print("\n2. DERIVED FIELD VALIDATION")
print("-" * 115)

customer_order_silver = silver_dataframes["customer_order"]

derived_field_exists = (
    "planning_horizon_status"
    in customer_order_silver.columns
)

derived_status = (
    "PASS" if derived_field_exists else "REVIEW"
)

validation_results.append(derived_status)

print(
    f"customer_order.planning_horizon_status exists"
    f"{' ' * 20}| {derived_status}"
)

if derived_field_exists:

    (
        customer_order_silver
        .groupBy("planning_horizon_status")
        .count()
        .orderBy("planning_horizon_status")
        .show(truncate=False)
    )


# ---------------------------------------------------------------
# 3. Validate preservation of due-date values
# ---------------------------------------------------------------

print("\n3. DUE-DATE PRESERVATION VALIDATION")
print("-" * 115)

bronze_orders = spark.table(
    f"{bronze_catalog}.{bronze_schema}.customer_order"
)

bronze_due_dates = (
    bronze_orders
    .select(
        "order",
        "due_date_days_in_the_planning_horizon"
    )
)

silver_due_dates = (
    customer_order_silver
    .select(
        F.col("order_id").alias("order"),
        "due_date_days_in_the_planning_horizon"
    )
)

due_date_difference_count = (
    bronze_due_dates
    .subtract(silver_due_dates)
    .union(
        silver_due_dates.subtract(bronze_due_dates)
    )
    .count()
)

due_date_status = (
    "PASS"
    if due_date_difference_count == 0
    else "REVIEW"
)

validation_results.append(due_date_status)

print(
    f"Bronze vs Silver due-date differences: "
    f"{due_date_difference_count} | {due_date_status}"
)


# ---------------------------------------------------------------
# 4. Validate preservation of main_alternative codes
# ---------------------------------------------------------------

print("\n4. MAIN-ALTERNATIVE DOMAIN PRESERVATION")
print("-" * 115)

bronze_main_alt = {
    row["main_alternative"]
    for row in spark.table(
        f"{bronze_catalog}.{bronze_schema}."
        f"product_production_alternative"
    )
    .select("main_alternative")
    .distinct()
    .collect()
}

silver_main_alt = {
    row["main_alternative"]
    for row in silver_dataframes[
        "product_production_alternative"
    ]
    .select("main_alternative")
    .distinct()
    .collect()
}

main_alt_status = (
    "PASS"
    if bronze_main_alt == silver_main_alt
    else "REVIEW"
)

validation_results.append(main_alt_status)

print(f"Bronze domain: {sorted(bronze_main_alt)}")
print(f"Silver domain: {sorted(silver_main_alt)}")
print(f"Domain preservation: {main_alt_status}")


# ---------------------------------------------------------------
# 5. Validate preservation of MTS/MTO domain
# ---------------------------------------------------------------

print("\n5. MTS/MTO DOMAIN PRESERVATION")
print("-" * 115)

bronze_mts_mto = {
    row["mts_mto"]
    for row in spark.table(
        f"{bronze_catalog}.{bronze_schema}.finished_product_type"
    )
    .select("mts_mto")
    .distinct()
    .collect()
}

silver_mts_mto = {
    row["mts_mto"]
    for row in silver_dataframes["finished_product_type"]
    .select("mts_mto")
    .distinct()
    .collect()
}

mts_mto_status = (
    "PASS"
    if bronze_mts_mto == silver_mts_mto
    else "REVIEW"
)

validation_results.append(mts_mto_status)

print(f"Bronze domain: {sorted(bronze_mts_mto)}")
print(f"Silver domain: {sorted(silver_mts_mto)}")
print(f"Domain preservation: {mts_mto_status}")


# ---------------------------------------------------------------
# 6. Final contract result
# ---------------------------------------------------------------

print("=" * 115)

passed = validation_results.count("PASS")
review = validation_results.count("REVIEW")

print(f"Validation controls executed: {len(validation_results)}")
print(f"PASS: {passed}")
print(f"REVIEW: {review}")

if review == 0:
    print(
        "SILVER TRANSFORMATION CONTRACT VALIDATION PASSED."
    )
else:
    print(
        "SILVER TRANSFORMATION CONTRACT REQUIRES REVIEW."
    )

print(
    "Silver Delta tables have NOT yet been persisted."
)

SILVER TRANSFORMATION — CONTRACT VALIDATION

1. CONTROLLED COLUMN RENAME VALIDATION
-------------------------------------------------------------------------------------------------------------------
customer_order                      | order -> order_id                                 | PASS
product_packing_machine             | finished_products -> finished_product                         | PASS
demand_plan                         | total_demand_m_adjusted_if_orders_are_greater_than_forecasts -> adjusted_total_demand_m                  | PASS
semi_finished_acquisition           | semi_finished_products_that_can_be_acquired_from_the_market -> semi_finished_product                    | PASS
semi_finished_acquisition           | maximum_purchasing_quantity_per_macroperiod_month_m -> max_purchase_quantity_per_month_m        | PASS
raw_material                        | maximum_purchasing_quantity_per_macroperiod_month_kg -> max_purchase_quantity_per_month_kg       | PASS

2. DERIVED FIEL

In [0]:
# Project #4 — Supply Chain Planning & Operations Intelligence
# Silver Layer — Step 13: Persist Silver Delta Tables
#
# Purpose:
# Persist the 21 validated Silver DataFrames as managed Delta tables
# in Unity Catalog.
#
# Bronze tables remain unchanged.

silver_catalog = "workspace"
silver_schema = "silver_sop"

print("SILVER TRANSFORMATION — PERSIST DELTA TABLES")
print("=" * 105)

written_silver_tables = []

for table_name in sorted(silver_dataframes.keys()):

    target_table = (
        f"{silver_catalog}.{silver_schema}.{table_name}"
    )

    (
        silver_dataframes[table_name]
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    written_silver_tables.append(target_table)

    print(
        f"WRITTEN | {target_table}"
    )

print("=" * 105)
print(
    f"Silver Delta tables written: "
    f"{len(written_silver_tables)}"
)
print(
    f"Target schema: "
    f"{silver_catalog}.{silver_schema}"
)

if len(written_silver_tables) == 21:
    print(
        "SILVER DELTA PERSISTENCE COMPLETE: "
        "21 tables written."
    )
else:
    raise ValueError(
        f"Expected 21 Silver tables to be written, "
        f"but wrote {len(written_silver_tables)}."
    )

SILVER TRANSFORMATION — PERSIST DELTA TABLES
WRITTEN | workspace.silver_sop.alternative_machine
WRITTEN | workspace.silver_sop.attainment_parameter
WRITTEN | workspace.silver_sop.customer_order
WRITTEN | workspace.silver_sop.demand_plan
WRITTEN | workspace.silver_sop.finished_product_economics
WRITTEN | workspace.silver_sop.finished_product_type
WRITTEN | workspace.silver_sop.machine
WRITTEN | workspace.silver_sop.machine_capacity
WRITTEN | workspace.silver_sop.packing_capacity
WRITTEN | workspace.silver_sop.packing_machine
WRITTEN | workspace.silver_sop.planning_parameters
WRITTEN | workspace.silver_sop.planning_periods
WRITTEN | workspace.silver_sop.product_inventory
WRITTEN | workspace.silver_sop.product_lot_size
WRITTEN | workspace.silver_sop.product_packing_machine
WRITTEN | workspace.silver_sop.product_production_alternative
WRITTEN | workspace.silver_sop.product_raw_material_requirement
WRITTEN | workspace.silver_sop.product_successor
WRITTEN | workspace.silver_sop.raw_material


In [0]:
# Project #4 — Supply Chain Planning & Operations Intelligence
# Silver Layer — Step 14: Verify Persisted Silver Delta Tables
#
# Purpose:
# Read the persisted Silver Delta tables back from Unity Catalog and
# reconcile them against the Bronze source.
#
# This validates persisted storage independently of the in-memory
# Silver DataFrames used during transformation.

from pyspark.sql import functions as F

bronze_catalog = "workspace"
bronze_schema = "bronze_sop"

silver_catalog = "workspace"
silver_schema = "silver_sop"

print("SILVER TRANSFORMATION — PERSISTED DELTA VALIDATION")
print("=" * 120)


# ---------------------------------------------------------------
# 1. Discover persisted Silver tables
# ---------------------------------------------------------------

silver_tables = spark.sql(
    f"SHOW TABLES IN {silver_catalog}.{silver_schema}"
).select("tableName").collect()

persisted_silver_tables = sorted(
    [row["tableName"] for row in silver_tables]
)

print(f"Persisted Silver tables found: {len(persisted_silver_tables)}")
print()


# ---------------------------------------------------------------
# 2. Reconcile persisted Silver tables to Bronze
# ---------------------------------------------------------------

total_bronze_rows = 0
total_silver_rows = 0
table_validation_results = []

for table_name in persisted_silver_tables:

    bronze_table = (
        f"{bronze_catalog}.{bronze_schema}.{table_name}"
    )

    silver_table = (
        f"{silver_catalog}.{silver_schema}.{table_name}"
    )

    bronze_exists = spark.catalog.tableExists(bronze_table)
    silver_exists = spark.catalog.tableExists(silver_table)

    if not bronze_exists or not silver_exists:
        status = "REVIEW"

        print(
            f"{table_name:<40} | "
            f"Bronze Exists: {bronze_exists} | "
            f"Silver Exists: {silver_exists} | "
            f"{status}"
        )

        table_validation_results.append(status)
        continue

    bronze_df = spark.table(bronze_table)
    silver_df = spark.table(silver_table)

    bronze_rows = bronze_df.count()
    silver_rows = silver_df.count()

    total_bronze_rows += bronze_rows
    total_silver_rows += silver_rows

    status = (
        "PASS"
        if bronze_rows == silver_rows
        else "REVIEW"
    )

    table_validation_results.append(status)

    print(
        f"{table_name:<40} | "
        f"Bronze Rows: {bronze_rows:<5} | "
        f"Silver Rows: {silver_rows:<5} | "
        f"Silver Cols: {len(silver_df.columns):<3} | "
        f"{status}"
    )


# ---------------------------------------------------------------
# 3. Validate persisted Silver customer_order schema
# ---------------------------------------------------------------

print("\nPERSISTED CUSTOMER_ORDER VALIDATION")
print("-" * 120)

silver_orders = spark.table(
    f"{silver_catalog}.{silver_schema}.customer_order"
)

expected_order_columns = [
    "order_id",
    "finished_product",
    "due_date_days_in_the_planning_horizon",
    "quantity_m",
    "sales_value_eur",
    "planning_horizon_status"
]

actual_order_columns = silver_orders.columns

order_schema_status = (
    "PASS"
    if actual_order_columns == expected_order_columns
    else "REVIEW"
)

print(f"Expected columns: {expected_order_columns}")
print(f"Actual columns:   {actual_order_columns}")
print(f"Customer-order schema: {order_schema_status}")


# ---------------------------------------------------------------
# 4. Validate persisted planning-horizon classification
# ---------------------------------------------------------------

print("\nPERSISTED HORIZON-STATUS DISTRIBUTION")
print("-" * 120)

(
    silver_orders
    .groupBy("planning_horizon_status")
    .count()
    .orderBy("planning_horizon_status")
    .show(truncate=False)
)

horizon_counts = {
    row["planning_horizon_status"]: row["count"]
    for row in (
        silver_orders
        .groupBy("planning_horizon_status")
        .count()
        .collect()
    )
}

horizon_status_pass = (
    horizon_counts.get("Before horizon start", 0) == 9
    and horizon_counts.get("Within horizon", 0) == 52
    and horizon_counts.get("Beyond horizon end", 0) == 0
)

horizon_validation_status = (
    "PASS"
    if horizon_status_pass
    else "REVIEW"
)

print(
    f"Horizon-status validation: "
    f"{horizon_validation_status}"
)


# ---------------------------------------------------------------
# 5. Final persisted-layer validation
# ---------------------------------------------------------------

print("=" * 120)

table_count_status = (
    "PASS"
    if len(persisted_silver_tables) == 21
    else "REVIEW"
)

row_total_status = (
    "PASS"
    if total_bronze_rows == total_silver_rows == 4559
    else "REVIEW"
)

all_table_rows_pass = all(
    status == "PASS"
    for status in table_validation_results
)

print(f"Silver table-count validation:    {table_count_status}")
print(f"Per-table row reconciliation:     {'PASS' if all_table_rows_pass else 'REVIEW'}")
print(f"Total Bronze records:             {total_bronze_rows}")
print(f"Total persisted Silver records:   {total_silver_rows}")
print(f"Total-row reconciliation:         {row_total_status}")
print(f"Customer-order schema validation: {order_schema_status}")
print(f"Horizon-status validation:        {horizon_validation_status}")

if all([
    table_count_status == "PASS",
    all_table_rows_pass,
    row_total_status == "PASS",
    order_schema_status == "PASS",
    horizon_validation_status == "PASS"
]):
    print(
        "PERSISTENCE VALIDATION PASSED: "
        "21 of 21 Silver Delta tables exist and reconcile successfully."
    )
else:
    print(
        "PERSISTENCE VALIDATION REQUIRES REVIEW."
    )

SILVER TRANSFORMATION — PERSISTED DELTA VALIDATION
Persisted Silver tables found: 21

alternative_machine                      | Bronze Rows: 734   | Silver Rows: 734   | Silver Cols: 3   | PASS
attainment_parameter                     | Bronze Rows: 3     | Silver Rows: 3     | Silver Cols: 3   | PASS
customer_order                           | Bronze Rows: 61    | Silver Rows: 61    | Silver Cols: 6   | PASS
demand_plan                              | Bronze Rows: 246   | Silver Rows: 246   | Silver Cols: 4   | PASS
finished_product_economics               | Bronze Rows: 67    | Silver Rows: 67    | Silver Cols: 4   | PASS
finished_product_type                    | Bronze Rows: 67    | Silver Rows: 67    | Silver Cols: 2   | PASS
machine                                  | Bronze Rows: 39    | Silver Rows: 39    | Silver Cols: 2   | PASS
machine_capacity                         | Bronze Rows: 312   | Silver Rows: 312   | Silver Cols: 4   | PASS
packing_capacity                         |

In [0]:
# Project #4 — Supply Chain Planning & Operations Intelligence
# Silver Layer — Step 15: Silver Transformation Completion Summary
#
# Purpose:
# Document the final validated state of the Silver transformation layer.

bronze_catalog = "workspace"
bronze_schema = "bronze_sop"

silver_catalog = "workspace"
silver_schema = "silver_sop"

silver_tables = spark.sql(
    f"SHOW TABLES IN {silver_catalog}.{silver_schema}"
).select("tableName").collect()

silver_table_names = sorted(
    [row["tableName"] for row in silver_tables]
)

total_silver_rows = sum(
    spark.table(
        f"{silver_catalog}.{silver_schema}.{table_name}"
    ).count()
    for table_name in silver_table_names
)

print("PROJECT #4 — SILVER TRANSFORMATION COMPLETION SUMMARY")
print("=" * 95)

print(f"Bronze source schema:        {bronze_catalog}.{bronze_schema}")
print(f"Silver target schema:        {silver_catalog}.{silver_schema}")
print(f"Bronze source tables:        21")
print(f"Silver Delta tables:         {len(silver_table_names)}")
print(f"Persisted Silver records:    {total_silver_rows}")

print()
print("DATA-QUALITY / STRUCTURAL VALIDATION")
print("-" * 95)

print("Exact duplicate rows:        0")
print("SQL NULL cells:              0")
print("Blank/whitespace values:     0")
print("Candidate key tests:         19/19 PASS")
print("Remaining grains reviewed:   2/2")
print("Referential tests:           16/16 PASS")
print("Planning-period controls:    4/4 PASS")

print()
print("CONTROLLED SILVER TRANSFORMATIONS")
print("-" * 95)

print("Controlled column renames:   6")
print("Derived analytical fields:   1")
print("Due-date values preserved:   YES")
print("main_alternative preserved:  YES — source codes 1/2")
print("MTS/MTO domain preserved:    YES")

print()
print("CUSTOMER ORDER HORIZON PROFILE")
print("-" * 95)

print("Before horizon start:        9")
print("Within horizon:              52")
print("Beyond horizon end:          0")

print()
print("PERSISTENCE VALIDATION")
print("-" * 95)

print("Silver tables validated:     21/21")
print("Bronze records:              4559")
print("Silver records:              4559")
print("Row reconciliation:          PASS")

print("=" * 95)

if (
    len(silver_table_names) == 21
    and total_silver_rows == 4559
):
    print("SILVER TRANSFORMATION STATUS: COMPLETE")
    print(
        "Validated Bronze data standardized and persisted as "
        "21 analysis-ready Silver Delta tables with full "
        "record-count reconciliation."
    )
else:
    print("SILVER TRANSFORMATION STATUS: REVIEW REQUIRED")

PROJECT #4 — SILVER TRANSFORMATION COMPLETION SUMMARY
Bronze source schema:        workspace.bronze_sop
Silver target schema:        workspace.silver_sop
Bronze source tables:        21
Silver Delta tables:         21
Persisted Silver records:    4559

DATA-QUALITY / STRUCTURAL VALIDATION
-----------------------------------------------------------------------------------------------
Exact duplicate rows:        0
SQL NULL cells:              0
Blank/whitespace values:     0
Candidate key tests:         19/19 PASS
Remaining grains reviewed:   2/2
Referential tests:           16/16 PASS
Planning-period controls:    4/4 PASS

CONTROLLED SILVER TRANSFORMATIONS
-----------------------------------------------------------------------------------------------
Controlled column renames:   6
Derived analytical fields:   1
Due-date values preserved:   YES
main_alternative preserved:  YES — source codes 1/2
MTS/MTO domain preserved:    YES

CUSTOMER ORDER HORIZON PROFILE
---------------------------